# 第 13 章 · 探索与利用：多臂老虎机与 ε-greedy

本章回答：不知道环境的概率时，只能自己去试。总挑"目前看起来最好的"有什么问题？怎么在"试新的"和"用好的"之间平衡？怎么一边试一边更新平均值？

配套讲义：仓库里 `cs285-deep-rl/notes/lessons/13-bandit.md`（讲解、动画、练习都在讲义里）。

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU。

In [ ]:
import numpy as np

np.set_printoptions(precision=3, suppress=True)  # 打印保留 3 位小数
print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 代码 | 跑出来 |
|---|---|---|
| 增量平均 | `incremental_mean([2, 4, 6, 8])` | `[2.0, 3.0, 4.0, 5.0]` |
| 贪心，200 次实验 × 1000 步 | `run_bandit(0.0, seed=s)` | 每步 1.105，最好机器 21% |
| ε-greedy | `run_bandit(0.1, seed=s)` | 每步 1.426，最好机器 89% |

## 1. 先把要用的东西定义好

这几格只是定义（环境、函数），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `incremental_mean`**（来自 `ch13_bandit.py`）：下面要用到的小工具。

In [ ]:
def incremental_mean(rewards):
    q, hist = 0.0, []
    for n, r in enumerate(rewards, start=1):
        q = q + (r - q) / n
        hist.append(q)
    return hist

**常量 `TRUE_MEANS`**（来自 `ch13_bandit.py`）

In [ ]:
TRUE_MEANS = np.array([1.0, 1.5, 0.5])

**函数 `run_bandit`**（来自 `ch13_bandit.py`）：下面要用到的小工具。

In [ ]:
def run_bandit(eps, steps=1000, seed=0, means=TRUE_MEANS):
    rng = np.random.default_rng(seed)
    Q, N = np.zeros(len(means)), np.zeros(len(means))
    total, best = 0.0, 0
    for _ in range(steps):
        a = rng.integers(len(means)) if rng.random() < eps else int(np.argmax(Q))
        r = rng.normal(means[a], 1.0)
        N[a] += 1
        Q[a] += (r - Q[a]) / N[a]
        total += r
        best += a == np.argmax(means)
    return total / steps, best / steps, Q

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("增量平均 [2, 4, 6, 8]：", incremental_mean([2, 4, 6, 8]))

In [ ]:
for eps in [0.0, 0.1]:
        res = [run_bandit(eps, seed=s) for s in range(200)]
        avg_r = np.mean([r[0] for r in res])
        pct = np.mean([r[1] for r in res])
        print(f"ε = {eps}：200 次实验平均每步奖励 {avg_r:.3f}，选到最好机器的比例 {pct:.0%}")

## 3. 代码里才看得到的

`np.argmax` 遇到并列时总是返回**编号最小**的那个，所以全 0 时贪心第一次一定拉 1 号机器，后面就可能一直困在 1 号。更公平的做法是在并列的里面随机挑（第 16 章的代码就是这样写的）。

## 4. 自己改一改

讲义里建议的几个小实验（数字是实跑出来的，改完对一对）：

1. 换 ε 跑 200 次实验：ε = 0.01 → 每步 1.306、最好机器 61%；ε = 0.1 → 1.426、89%；ε = 0.5 → 1.246、66%。试太少学不到，试太多又浪费，中间最好。
2. 把 `steps` 改成 5000，再比一次 ε = 0.01 和 ε = 0.1，看谁追上来了。

讲义末尾的练习题也可以在下面的空格子里用代码验算：先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 不知道 p 就去试；只挑最好会被第一印象困住；
> 利用拿眼前，探索找更好，一次只能做一样；
> ε-greedy：九成贪心一成乱试；
> 新估计 = 旧估计 + 步长 × (目标 − 旧估计)，全书都是这句话。